# Title: Improving detection of fake news using dataset augmentation by generating more fake articles using LLM and using RL q-learning to determine optimal amount of data augmentation.

#### Members' Names or Individual's Name: Alexander Eliseev , Divya Bharathi Kannappan , Ahmed Mohamed
####  Emails: aeliseev@torontomu.ca, divya.kannappan@torontomu.ca (add your emails)

# Introduction:

#### Problem Description:

The unchecked spread of misinformation across digital platforms has evolved into a systemic threat, making the detection of "fake news" one of the most pressing challenges for modern information systems. While machine learning offers a promising defense, these models are only as effective as the data used to train them. Currently, we face a significant bottleneck: the lack of balanced, high-fidelity datasets required to build truly resilient detection systems.

#### Context of the Problem:

In practice, the data we encounter is rarely symmetrical. Real-world datasets suffer from a severe class imbalance, where legitimate news articles far outnumber verified instances of misinformation. This scarcity prevents models from learning the nuanced features of deceptive content, leading to a "blind spot" where the system generalizes poorly to new, emerging threats. Furthermore, because misinformation is constantly evolving in style and structure, models trained on static, limited data quickly become obsolete. The biggest problem remains is being able to identify real misinformation and fake news published by verified sources. Unverified sources can be simply assumed to be predominantly fake and simply black listed but, it is misinformation created by long standing and verified sources which remains the problem and needs to be kept in check. Luckily most articles published by verified sources are going to be curated or peer reviewed multiple times and as such the number of fake information articles will nautrally be far lower than of real ones. Classification training however wants all classes to be in equal amounts for the best performance as such the limiting factor will be the number of fake news in the training pool. Since no more real fake news are available online the remaining solution is to augment the available ones with more artificially generated ones. In this project this will be achieved using NLP.
#### Limitation About other Approaches:

Traditional detection frameworks typically fall short in two key areas:

Static Data Dependency: Most approaches rely strictly on pre-existing labeled datasets, failing to account for the inherent scarcity of fake news samples.

Lack of Generative Innovation: There is a notable absence of generative modeling to bridge the data gap. By failing to synthesize diverse training examples, existing models are prone to overfitting—they become excellent at identifying past fake news but fail to recognize the misinformation of tomorrow.

#### Solution:

The proposed method addresses the problem of limited and imbalanced fake news data by generating high-quality synthetic fake articles using LLMs guided by extracted topics and named entities. This augmented dataset improves the model’s ability to learn diverse fake news patterns, leading to better classification performance and generalization. Furthermore the augmented dataset allows for a greater ceiling of real news articles to be used in training without causing a bias in the model.


# Background

| Reference            | Explanation                                                | Dataset/Input       | Weakness                            |
| -------------------- | ---------------------------------------------------------- | ------------------- | ----------------------------------- |
| BERT (Devlin et al.) | Transformer-based model for text classification            | Large-scale corpora | High computational cost             |
| LSTM Networks        | Sequence-based deep learning model for text classification | Text datasets       | Poor long-range dependency handling |
| LDA (Blei et al.)    | Topic modeling using bag-of-words                          | Text corpus         | Ignores semantic meaning            |
| NER (spaCy)          | Extracts named entities such as people                     | Raw text            | May introduce noise                 |
| LLMs (LLAMA 3)       | Generates human-like text based on prompts                 | Prompt-based input  | Risk of synthetic bias              |


# Methodology

This project proposes a data augmentation-based approach for imporving fake news detection by combining topic modeling, named entity recognition (NER), large language models (LLMs), and reinforcement learning. Fake news articles are first scrapped off the internet and then processed using spaCy NER to extract named entities from the articles. These named entities are then removed from the texts along with generic words, punctuation and other noise to avoid them getting identified as topics. After which the remaining words of each article are lemmatized and kept as tokens. Finally LDA topic modelling is performed on the tokenized set of the fake articles to extract topics from them. Due to the variety of fake news and a considerable training pool of 1835 articles a total of 200 different topics could be reliably extracted. Finally, the named entities and topics are randomly selected and used as prompts for an LLM to generate a news article involving them. In this project LLAMA 3 is used.

With the data augmentation complete, an LSTM classifier is trained to classsify fake/real news and to set a baseline it is trained using only available fake and real news with the real news far outnumbering the fake ones. To improve the accuracy of the model by balancing the training data,the fake news are augmented with their artifically generated counterparts.

Lastly RL is used to determine the optimal amount of data augmentation should be done for the maximum possible accuracy from the classifier. Too much augmentation can result in bias towards the generated fake news articles and too little of them would not have an affect on the accuracy.

# Implementation

### Step 1: Data Collection

Fake and real news articles are collected using the GossipCop dataset which has thousands of links to fake and real articles posted from various new sources on the internet. This was one only English lanugage dataset of fake and real news used in the original research paper. Python scripts are used to read the article URLs from the GossipCop dataset and then use the native Newspaper library in Python to scrape the article text from the internet. 

* Real news articles are downloaded and saved into a training directory
* Fake news articles are similarly collected and stored into a training folder
* 400 Fake and 400 Real articles are stored seperately in a testing directory
* Articles with insufficient content are filtered out, articles that have been taken down are also ignored. Roughly 40% of the GossipCop dataset is no longer valid and can't be retrieved.

---

### Step 2: Data Preprocessing

The collected fake news articles undergo preprocessing, including:

* Lowercasing text
* Removing punctuation and stopwords
* Lemmatization using NLTK
* Tokenization of words
* Named Entity Recognition (NER) using spaCy is applied to extract names of people from the articles.

*Note above preprocessing operations are done as needed directly things like classifier training or LDA for prompt engineering for generating new fake articles using LLAMA 3 and so on. The original downloaded article texts remain untouched in the training folders besides having things like <br > removed.

---

### Step 3: Topic Extraction using LDA

Latent Dirichlet Allocation (LDA) is applied to the cleaned fake news dataset to identify underlying topics. These topics represent common themes present in fake news articles and is used for prompt topics to feed into LLAMA 3 LLM to generate more fake news articles.

* Articles are converted into bag-of-words format
    * After NER is used to remove all named entities, lower casing text, removing stopwords/noise and Lemmatizing words
* LDA model is trained to extract multiple topics
* Each topic consists of a set of representative words that relate to one another
* Total of 200 different topics could be extracted whilst maintaining their quality and them not being too generic

---

### Step 4: Synthetic Fake News Generation

To address class imbalance, synthetic fake news articles are generated using a Large Language Model (LLAMA 3).

* Random combinations of extracted topics and named people are selected
* These are used as prompts to guide the LLM
* The model generates realistic fake news articles based on the prompts
* Generated articles are filtered to ensure quality before being saved, removing extra text that the LLM may have generated by accident

Example prompt given to LLAMA 3:
    You are a news article generator. 

    Write an article given the following people:
    ['Neil Sean', 'Magda', "Katy Perry's", 'Maria Menounos', 'Guy P. Jones']

    And make up a logical story or argument or drama or event involving the listed people and these topic words:
    ['think', 'last', 'lexus', 'crash', "'re", 'former', 'episode', 'highway', 'paparazzo','moment']


    Write 3-5 Paragraphs using the topic words and the people in a logical manner. Try to avoid repeats.
    You may choose to make it in gossip news style or make it more formal rumours. Or a formal event that you are reporting on.
    It may also be an anouncement or just news.

    Make it anywhere between 600-1200 words.

    Commit to the chosen style.

So setting up the fake news generation took a fair bit of trial and error. As a whole, it is very resource heavy especially if going for a high degree of quality. The inital setup used was the prompt above where the model was given freedom to choose the style at random on its own and given a token limit of 1500. The first batch of articles were generated overnight, ~14 hours worth of time and a mere 43 were made. They are numberered 0-42.txt and can be viewed in the Data/train/GeneratedFake folder. This initial batch was upon inspection, of very believable quality and good variety and would probably be the best for training quality however. The rate at which they were being made was just too slow. The generation setup would push the hardware setup to use 32/32GB of RAM and the RTX 4060 GPU to 100% utilization. And even still it was like 1 article every 20 minutes. Taking into account that there are 1800 real fakes and ~6000 real articles in the training set that can be used. 80 articles/day is just not practical. To improve the rate of generating the fake articles, max new tokens was reduced down to 600, and number of returns was upped to 3, 10 was also tested. So with the LLM the biggest time sink in generation is inferencing the prompt to capture the context of what should be generated. Once the context is setup, the generation of the text itself is relatively fast. Based on observation, it takes about the same amount of time to write 10 slightly different articles from the same prompt as it does to generate 2 articles from 2 different prompts, assuming same token size for the articles. With this said, the prompt was simplified to reduce decision making for the LLM and also token limit was reduced to 600. And about 3 similar articles would now be trained for every prompt. This would reduce their overall quality as training data as there is less difference between the articles and the articles themselves may have less diverse vocabulary but this decrease in quality would be offset by the fact that the bias in the training dataset would be reduced as more fake news articles were present and more real articles could be used in training too. Either way after the changes, about 4-5x as many could be made in the same amount of time. Tragically on the 2nd night of generation the computer crashed, either from the utilization or possibly because of the power flickering from the thunderstorm outside so rather than having 300 the next morning only 75 more were made thoughout the night. Either way a lot better rate.



---

### Step 5: Classification Model

A Long Short-Term Memory (LSTM) model is used to classify news articles as real or fake.

* Input: Batches of real fake, syntehtic fake and real news articles
* Output: Binary classification (real/fake)
* The model learns linguistic patterns and contextual cues

LSTM is a lot lighter than transformers in terms of memory which is why it was chosen over using BERT + Transformer setup. And even it required batch loading and would crash and run out of memory on a 32GB setup if the entire dataset of only ~10k articles would be loaded. To avoid the memory issues batching implemented which solved the problem and allowed it to train. Inital results were good, with the original training set of ~1800 fakes and ~6k real articles resulting in a final model that would have a ~52% accuracy. After the introduction of just 50 aritifical articles the accuracy score would reach ~57% and showed promising results. 

---

### Step 6: Reinforcement Learning Optimization

Reinforcement learning is used to determine the optimal number of synthetic fake news samples to include in training.

* Action: Number of synthetic samples added
* Reward: Validation accuracy of the classifier
* Goal: Maximize classification performance

Reinforcement learning and using Q-learning was the proposed solution in the reasearch paper. It is certainly a novel idea as this is something that could probably be implemented simply using a regression model on a dataset of accuracy vs number of synthetic fake articles used in training. Which when plotted would probably look like a quadratic function as accuracy would initially increase benefitting from the synthetics and then start to decrease once more as they become a bias in and of themselves outweighing in value the actual real fakes and overfitting the model. But, using Q-learning to converge by adding or taking away X aritifical articles every step from the training pool and seeing the accuracy from the adjustment certainly works too. Not sure how effecient it is resource wise not to mention that it would have limited exploration and could get stuck but, it could be more efficient if it converges to the global maxima right away.

---

### Key Idea

The key idea of this project is to improve fake news detection by augmenting the dataset with high-quality synthetic data generated using LLMs, rather than relying solely on limited real-world data. Below, is the final demo of the implementation described above. The demo consists of 3 parts, first testing the classifier with no data augmentation, then with all available augmented data included and lastly a run of RL to compute the optimal amount of data augmentation that should be used. Then the experiment is repeated but with a smaller initial training dataset of real and fake articles from the internet. Comments below explain the rationale behind each part of the experiment further.


In [19]:
#First code cell runs the LSTM classifier using no generated fake articles but, using all 1835 real fake articles and ~6000 real articles from the internet.
#This sets a baseline accuracy of what is to be expected from the classifier if only available data is used.
import sys
import importlib
sys.path.append("src")
import TrainAndEvaluateClassifier
import TrainAndEvaluateLSTM
import ReinforcementLearning
importlib.reload(TrainAndEvaluateClassifier)
importlib.reload(TrainAndEvaluateLSTM)
importlib.reload(ReinforcementLearning)
from TrainAndEvaluateClassifier import TrainAndEvaluateClassifer
from TrainAndEvaluateLSTM import TrainAndEvaluateLSTM
from ReinforcementLearning import train_q_learning

def run_single_experiment(): #Function also available in main.py
    print("=== Running LSTM Classifier, Training using all fake and real articles available. No generated fake articles are used. ===")

    data = TrainAndEvaluateClassifer(
        NumberOfArtificialArticlesTouse=0,
        UseAll=1
    )

    accuracy = TrainAndEvaluateLSTM(data)
    print(f"\nBaseline accuracy: {accuracy:.4f}")

run_single_experiment()

=== Running LSTM Classifier, Training using all fake and real articles available. No generated fake articles are used. ===
Train size: 8300
Test size: 800
Epoch 1, Loss: 35.9119
Epoch 2, Loss: 33.1770
Epoch 3, Loss: 31.3933
Epoch 4, Loss: 28.1001
Epoch 5, Loss: 23.7366
Test Accuracy: 0.5350

Baseline accuracy: 0.5350


In [21]:
#As can be seen in the cell above, the LSTM model struggles to classify real from fakes only achieving an accuracy of ~52% when trained using 
#just the original fake and real news articles collected online. Next this cell will show what happens when all of the generated fake news articles are
#added into the training pool.
def run_single_experiment(): #Function also available in main.py
    print("=== Running LSTM Classifier, Training using all fake and real articles available. No generated fake articles are used. ===")

    data = TrainAndEvaluateClassifer(
        NumberOfArtificialArticlesTouse=100000, #Any number that exceeds their max will just cause all to be used.
        UseAll=1
    )

    accuracy = TrainAndEvaluateLSTM(data)
    print(f"\nAccuracy using all available source and augmented data: {accuracy:.4f}")

run_single_experiment()

=== Running LSTM Classifier, Training using all fake and real articles available. No generated fake articles are used. ===
Train size: 8547
Test size: 800
Epoch 1, Loss: 38.8953
Epoch 2, Loss: 35.9749
Epoch 3, Loss: 34.0399
Epoch 4, Loss: 30.5731
Epoch 5, Loss: 26.1578
Test Accuracy: 0.5675

Accuracy using all available source and augmented data: 0.5675


In [ ]:
#As we can see above, the accuracy of the model has increased by about ~3% which makes sense considering that only about ~250 generated articles are
#actually used to augment the fake news articles portion of the training data set. This still means there are about ~2050 fake articles to ~6000 real ones
#Which is better than 1835 vs ~6k but its still a very large discrepency in the number of training examples between the two classes. So accuracy
#only marginally improves. This cell will now run to calculate the optimal amount of articles to add out of these ~250:
#Its pretty moot/redundant because of how large the desparity in the number real/fake + gen fake articles remains pretty much all of the
#artificial articles end up being useful.
def run_rl_optimization():
    print("\n=== Running Q-Learning Optimization Using All Available Real, Generated Fake and Fake articles for classifer training. ===")

    best = train_q_learning()

    print(f"\nOptimal number of synthetic articles: {best}")

run_rl_optimization()


=== Running Q-Learning Optimization Using All Available Real, Generated Fake and Fake articles for classifer training. ===

=== Episode 1 ===
Train size: 8430
Test size: 800
Epoch 1, Loss: 36.8522
Epoch 2, Loss: 34.6023
Epoch 3, Loss: 32.3034


KeyboardInterrupt: 

# Conclusion

This project presents a novel approach to fake news detection by combining NLP techniques, generative AI, and reinforcement learning. By generating synthetic fake news using LLMs guided by topics and named entities, the dataset was effectively augmented, leading to improved classification performance as shown by the accuracy of the model imrpoving. Showing that limited amounts of training data can be augmented to improve the overall performance of the final classifier. And that augmented data be it artificially created is still viable and useful for training. Adding even a couple hundred generated fake articles helped improve the models overall accuracy by a percent or two even though the gap between the number of fake and real articles was only marginically reduced as the real ones outnumbered the fake ones by ~4.2 intially and still do by ~4000. However, as shown in the second expirement on the reduced dataset, the accuracy goes up a lot more substantially.

### Limitations

* Synthetic data may introduce bias due to LLM writing patterns
* LSTM models are less effective compared to transformer-based architectures
* Topic modeling using LDA may not capture deep semantic relationships

### Future Work

* Replace LSTM with transformer-based models such as BERT
* Use advanced topic modeling techniques like BERTopic
* Improve prompt engineering for better synthetic data generation and find a way to faster generate quality fake articles
* Apply adversarial filtering to remove low-quality generated samples


# References:

[1]: Zhao Tong, Yimeng Gu, Huidong Liu, Qiang Liu, Shu Wu, Haichao Shi, Xiao-Yu Zhang, "Generate First, Then Sample: Enhancing Fake News Detection with LLM-Augmented Reinforced Sampling", Conference/Preprint, 2024.

[2]: Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova, "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding", NAACL-HLT, 2019.

[3]: David M. Blei, Andrew Y. Ng, Michael I. Jordan, "Latent Dirichlet Allocation", Journal of Machine Learning Research, Vol. 3, 2003.

[4]: Kai Shu, Amy Sliva, Suhang Wang, Jiliang Tang, Huan Liu, "Fake News Detection on Social Media: A Data Mining Perspective", ACM SIGKDD Explorations Newsletter, Vol. 19, Issue 1, 2017.

[5]: Meta AI, "LLaMA 3: Open and Efficient Foundation Language Models", 2024.